# JuaKazi SW Bias Classifier v4 — Fine-tune + Upload Dataset + Model Card

All-in-one notebook: trains the classifier, pushes model + dataset + model card to HuggingFace.

**Before running:**
1. Accelerator → **GPU T4 x2** (right sidebar → Session options)
2. Add dataset: right sidebar → Input → Add input → Datasets → `juakazi-sw-ground-truth`
3. Add HF token: right sidebar → Secrets → `HF_TOKEN` (must be a **write** token)

**Expected time:** ~75 min  
**Expected SW F1:** 0.82–0.90

## Step 1 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {gpu}  ({mem:.1f} GB VRAM)')
else:
    raise RuntimeError('No GPU — go to Session options and select T4 x2')

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 accelerate huggingface_hub datasets

## Step 3 — HuggingFace login

In [ ]:
import os
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
print(f'Token starts with: {hf_token[:8]}...')

login(token=hf_token, add_to_git_credential=False)

info = whoami(token=hf_token)
print(f"Logged in as: {info['name']}")
print(f"Orgs: {[o['name'] for o in info.get('orgs', [])]}")

## Step 4 — Load dataset

CSV is mounted automatically from the Kaggle dataset you added.

In [ ]:
import csv, os, glob

matches = glob.glob('/kaggle/input/**/ground_truth_sw_v5.csv', recursive=True)
if not matches:
    raise FileNotFoundError(
        'CSV not found under /kaggle/input/\n'
        'Add it via: right sidebar → Input → Add input → Datasets → juakazi-sw-ground-truth'
    )
CSV_PATH = matches[0]
print(f'Found CSV at: {CSV_PATH}')

def load_dataset_rows(path):
    texts, labels = [], []
    skipped = 0
    with open(path) as f:
        for row in csv.DictReader(f):
            text = row.get('text', '').strip()
            bias = row.get('has_bias', '').lower().strip()
            if not text or bias not in ('true', 'false'):
                skipped += 1
                continue
            texts.append(text)
            labels.append(1 if bias == 'true' else 0)
    print(f'Loaded {len(texts):,} rows ({skipped} skipped)')
    print(f'  BIAS=1: {sum(labels):,}  NEUTRAL=0: {len(labels)-sum(labels):,}')
    return texts, labels

texts, labels = load_dataset_rows(CSV_PATH)

## Step 5 — Upload dataset to HuggingFace

In [ ]:
import pandas as pd
from datasets import Dataset

print('Loading full CSV for HF upload...')
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')

ds = Dataset.from_pandas(df)
ds.push_to_hub(
    'juakazike/sw-ground-truth',
    token=hf_token,
    private=False,
    commit_message='Upload JuaKazi Swahili ground truth v5 — 64,723 rows',
)
print('Dataset live: https://huggingface.co/datasets/juakazike/sw-ground-truth')

## Step 6 — Tokenize, oversample, and split

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset as TorchDataset
import torch, random

MODEL_ID = 'Davlan/afro-xlmr-base'

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Stratified split
combined = list(zip(texts, labels))
random.seed(42)
random.shuffle(combined)
texts, labels = zip(*combined)

split = int(len(texts) * 0.8)
train_texts, val_texts   = list(texts[:split]), list(texts[split:])
train_labels, val_labels = list(labels[:split]), list(labels[split:])
print(f'Train: {len(train_texts):,}   Val: {len(val_texts):,}')

# Oversample minority (BIAS) to 25% of training set
bias_texts    = [t for t, l in zip(train_texts, train_labels) if l == 1]
neutral_texts = [t for t, l in zip(train_texts, train_labels) if l == 0]
target_bias   = len(neutral_texts) // 4
if len(bias_texts) < target_bias:
    extra = random.choices(bias_texts, k=target_bias - len(bias_texts))
    train_texts  = train_texts  + extra
    train_labels = train_labels + [1] * len(extra)
print(f'After oversampling — Train: {len(train_texts):,}  BIAS: {sum(train_labels):,}  NEUTRAL: {len(train_labels)-sum(train_labels):,}')

class BiasDataset(TorchDataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = BiasDataset(train_texts, train_labels)
val_dataset   = BiasDataset(val_texts,   val_labels)
print('Datasets ready.')

## Step 7 — Load model, WeightedTrainer, configure training

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np

print(f'Loading model: {MODEL_ID}')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.config.id2label = {0: 'NEUTRAL', 1: 'BIAS'}
model.config.label2id = {'NEUTRAL': 0, 'BIAS': 1}
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M')

n_neg = train_labels.count(0)
n_pos = train_labels.count(1)
pos_weight = n_neg / n_pos
print(f'Class weight for BIAS: {pos_weight:.1f}x')

class WeightedTrainer(Trainer):
    """CrossEntropyLoss weighted for class imbalance."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = torch.tensor([1.0, pos_weight], device=logits.device, dtype=logits.dtype)
        loss = torch.nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1}

training_args = TrainingArguments(
    output_dir='/kaggle/working/sw_bias_classifier_v4',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_steps=300,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to='none',
    dataloader_num_workers=2,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
print('Trainer ready.')

## Step 8 — Train

~75 min on T4 x2. Watch `eval_f1` — target 0.82+ by epoch 3.

In [ ]:
trainer.train()
print('Training complete.')

## Step 9 — Evaluate

In [ ]:
metrics = trainer.evaluate()
print('\n=== Validation metrics ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## Step 10 — Save model locally

In [ ]:
OUTPUT_DIR = '/kaggle/working/sw_bias_classifier_v4_final'
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

size_mb = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR, f))
    for f in os.listdir(OUTPUT_DIR)
) / 1e6
print(f'Saved to {OUTPUT_DIR}  ({size_mb:.0f} MB)')

## Step 11 — Push model to HuggingFace

In [ ]:
from huggingface_hub import HfApi

HF_REPO = 'juakazike/sw-bias-classifier-v4'
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, private=False, token=hf_token)

model.push_to_hub(HF_REPO, token=hf_token)
tokenizer.push_to_hub(HF_REPO, token=hf_token)
print(f'Model pushed: https://huggingface.co/{HF_REPO}')

## Step 12 — Push model card

In [ ]:
# Pull actual metrics from training
f1  = metrics.get('eval_f1', 0)
prec = metrics.get('eval_precision', 0)
rec  = metrics.get('eval_recall', 0)

model_card = f"""---
language:
- sw
license: apache-2.0
tags:
- text-classification
- gender-bias
- swahili
- afro-xlmr
- east-africa
datasets:
- juakazike/sw-ground-truth
metrics:
- f1
- precision
- recall
base_model: Davlan/afro-xlmr-base
---

# JuaKazi Swahili Gender Bias Classifier v4

Fine-tuned [afro-xlmr-base](https://huggingface.co/Davlan/afro-xlmr-base) for binary gender bias detection in Swahili text.

Part of the [JuaKazi Gender Sensitization Engine](https://github.com/Stella-Achar-Oiro/gender-sensitization-engine) — the only tool in East Africa that detects, corrects, and explains gender bias in African-language text.

## Validation Metrics (v4)

| Metric | Score |
|---|---|
| F1 | {f1:.3f} |
| Precision | {prec:.3f} |
| Recall | {rec:.3f} |

## Labels

| ID | Label |
|---|---|
| 0 | NEUTRAL |
| 1 | BIAS |

## Training Details

- **Base model:** Davlan/afro-xlmr-base
- **Training data:** 64,723 Swahili rows — [juakazike/sw-ground-truth](https://huggingface.co/datasets/juakazike/sw-ground-truth)
- **Oversampling:** Minority class (BIAS) oversampled to 25% of training set
- **Class weighting:** WeightedTrainer with CrossEntropyLoss
- **Epochs:** 5 · **LR:** 1e-5 · **Batch:** 16/32 · **Hardware:** Kaggle T4 x2

## Usage in JuaKazi

Stage 2 fallback — Swahili only, warn-only, never sets `has_bias_detected=True` directly.

Set in HuggingFace Space secrets:
```
JUAKAZI_ML_MODEL = juakazike/sw-bias-classifier-v4
JUAKAZI_ML_THRESHOLD = 0.75
```

## Quick Start

```python
from transformers import pipeline
classifier = pipeline("text-classification", model="juakazike/sw-bias-classifier-v4")
classifier("Daktari wa kiume alipima mgonjwa.")
# [{{'label': 'BIAS', 'score': 0.89}}]
```

## Limitations

- Swahili only (Kenya + Tanzania). No Sheng/Uganda coverage.
- Cohen's Kappa not yet measured — 2nd annotator not yet recruited.
- Ambiguous phrases like `Watoto wa Kike` may fire in advocacy contexts.
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo='README.md',
    repo_id=HF_REPO,
    token=hf_token,
    commit_message='Add model card with actual training metrics',
)
print(f'Model card pushed: https://huggingface.co/{HF_REPO}')